# AI-Driven Seasonal Crop Planning & Income Risk Forecasting
**Anantapur District, AP | Dept. of AI & ML | Third Review 2026**  
**Team: Seema Minds | MSAI24R003**

---
This notebook walks through the complete pipeline:
1. Data Loading & Exploration
2. Feature Engineering
3. LSTM Price Forecasting
4. Regression Yield Estimation
5. Risk Scoring
6. Profit Calculation & Crop Ranking
7. Visualizations


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120
print('✅ Imports OK')

## 1. Data Loading & Exploration

In [ ]:
from src.preprocessor import load_data, engineer_features

df = load_data()
print(f'Shape: {df.shape}')
print(f'Crops: {df.crop.unique()}')
print(f'Date range: {df.date.min()} → {df.date.max()}')
df.head(6)

In [ ]:
# Summary statistics per crop
df.groupby('crop')[['price_per_q','yield_q_ha','net_profit','rainfall_mm']].describe().round(1)

In [ ]:
# Price history plot
from src.visualizer import plot_price_history

for crop in ['Groundnut','Tomato','Cotton']:
    fig = plot_price_history(crop)
    plt.show()

## 2. Feature Engineering

In [ ]:
df_fe = engineer_features(df)
print(f'Features after engineering: {df_fe.shape[1]} columns')
print(df_fe.columns.tolist())
df_fe[df_fe.crop=='Groundnut'][['date','price_per_q','price_ma3','price_ma6','price_std6']].tail(8)

## 3. LSTM Price Forecasting

In [ ]:
from src.lstm_model import predict_next_price
import json
from pathlib import Path

print('Price Forecasts (next 3 months):')
for crop in ['Groundnut','Tomato','Cotton']:
    preds = predict_next_price(crop, n_steps=3)
    print(f'  {crop:12}: {[f"₹{p:,.0f}" for p in preds]}')

# Show metrics
print('\nModel Performance:')
for crop in ['Groundnut','Tomato','Cotton']:
    mfile = Path('../models') / f'metrics_lstm_{crop.lower()}.json'
    m = json.loads(mfile.read_text())
    print(f'  {crop:12}: RMSE={m["rmse_pct"]}%  MAE={m["mae_pct"]}%')

In [ ]:
# Forecast visualization
from src.visualizer import plot_forecast

for crop in ['Groundnut','Tomato','Cotton']:
    preds = predict_next_price(crop, n_steps=3)
    fig = plot_forecast(crop, preds)
    plt.show()

## 4. Yield Estimation (Regression)

In [ ]:
from src.regression_model import predict_yield
import json
from pathlib import Path

print('Yield Estimates (Kharif, 95mm rainfall):')
for crop in ['Groundnut','Tomato','Cotton']:
    y = predict_yield(crop, rainfall_mm=95, area_ha=90000, price_per_q=6000, month=8)
    print(f'  {crop:12}: {y:.2f} Q/Ha')

print('\nRegression Model Performance:')
for crop in ['Groundnut','Tomato','Cotton']:
    mfile = Path('../models') / f'metrics_reg_{crop.lower()}.json'
    m = json.loads(mfile.read_text())
    print(f'  {crop:12}: R²={m["r2"]}  Best={m["best_model"]}')

## 5. Risk Scoring Engine

In [ ]:
from src.risk_engine import compute_all_risks, risk_breakdown_text

risks = compute_all_risks()
for crop, r in risks.items():
    print(f'\n{crop}')
    print(risk_breakdown_text(r))

In [ ]:
# Risk breakdown chart
from src.visualizer import plot_risk_breakdown, plot_volatility_heatmap

fig = plot_risk_breakdown(risks)
plt.show()

fig = plot_volatility_heatmap()
plt.show()

## 6. Profit Calculation & Crop Ranking

In [ ]:
from src.lstm_model import predict_next_price
from src.regression_model import predict_yield
from src.ranking_engine import rank_crops, recommendation_text

crop_data = []
for crop in ['Groundnut','Tomato','Cotton']:
    preds = predict_next_price(crop, n_steps=3)
    price = sum(preds) / len(preds)
    yld   = predict_yield(crop, 95, 90000, price)
    crop_data.append({
        'crop': crop,
        'forecasted_price': price,
        'estimated_yield':  yld,
        'risk_score':       risks[crop]['risk_score'],
    })

ranked = rank_crops(crop_data)
print(recommendation_text(ranked))

In [ ]:
# Show ranking table
import pandas as pd
rows = []
for r in ranked:
    rows.append({
        'Rank': r['rank_label'],
        'Crop': r['crop'],
        'Forecasted Price': f"₹{r['price_per_q']:,.0f}",
        'Yield (Q/Ha)': r['yield_q_ha'],
        'Net Profit (₹/Ha)': f"₹{r['net_profit']:,.0f}",
        'Risk Score': r['risk_score'],
        'Final Score': r['final_score'],
    })
pd.DataFrame(rows)

In [ ]:
# Profit comparison + ranking charts
from src.visualizer import plot_profit_comparison, plot_ranking_summary

fig = plot_profit_comparison(ranked)
plt.show()

fig = plot_ranking_summary(ranked)
plt.show()

## 7. Full Evaluation Report
Run the complete evaluation across all modules.

In [ ]:
from src.evaluator import full_report
full_report()

---
## Launch the Web UI
Run the cell below — then open **http://localhost:7860** in your browser.

In [ ]:
# Uncomment to launch
# import subprocess
# subprocess.Popen(['python', '../ui/app.py'])
print('Run: python ui/app.py  → open http://localhost:7860')